# 05d - Spectral Waterfall
Run `05-00-shared-data.ipynb` first to load data.

In [1]:
%run 05-00-shared-data.ipynb
from utilities import (
    apply_publication_style, save_fig, SINGLE_COL_IN, 
    FULL_COL_IN, MAX_H_IN, MM, PROC_COLORS, proc_color, 
    ENERGY_PROCESSES, HEAT_RELEASE_PROCESSES, 
    HEAT_CONSUME_PROCESSES, stn_label)
import os
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import numpy as np
apply_publication_style()

Zarr store: /Users/schimmel/data/cosmo-specs/meteograms/cs-eriswil__20260304_110254/Meteogram_cs-eriswil__20260304_110254_nVar136_nMet3_nExp5.zarr
[########################################] | 100% Completed | 14.00 s
Shared data loaded successfully.
Created animated GIF: ./output/05/spectral_waterfall_number_exp2_stn_all_ALLBB_evolution.gif


In [2]:
from utilities import allocate_resources
c1,c2 = allocate_resources(n_cpu=64, m=32)

2026-03-10 19:01:33,880 - tornado.application - ERROR - Exception in callback functools.partial(<bound method IOLoop._discard_future_result of <tornado.platform.asyncio.AsyncIOMainLoop object at 0x308e09110>>, <Task finished name='Task-24278' coro=<SpecCluster._correct_state_internal() done, defined at /opt/homebrew/Caskroom/miniconda/base/envs/voodoo_tutorial/lib/python3.11/site-packages/distributed/deploy/spec.py:352> exception=FileNotFoundError(2, 'No such file or directory')>)
Traceback (most recent call last):
  File "/opt/homebrew/Caskroom/miniconda/base/envs/voodoo_tutorial/lib/python3.11/site-packages/tornado/ioloop.py", line 750, in _run_callback
    ret = callback()
          ^^^^^^^^^^
  File "/opt/homebrew/Caskroom/miniconda/base/envs/voodoo_tutorial/lib/python3.11/site-packages/tornado/ioloop.py", line 774, in _discard_future_result
    future.result()
  File "/opt/homebrew/Caskroom/miniconda/base/envs/voodoo_tutorial/lib/python3.11/site-packages/distributed/deploy/spec.py

#!/usr/bin/env bash

#SBATCH -J dask-worker
#SBATCH -p compute
#SBATCH -A bb1376
#SBATCH -n 1
#SBATCH --cpus-per-task=64
#SBATCH --mem=30G
#SBATCH -t 02:00:00
#SBATCH --output=./logs/%j.out
#SBATCH --error=./logs/%j.err
#SBATCH --propagate=STACK
source ~/.bashrc
conda activate pcpaper_env
export OMP_NUM_THREADS=1
export MKL_NUM_THREADS=1
export OPENBLAS_NUM_THREADS=1
export VECLIB_MAXIMUM_THREADS=1
export NUMEXPR_NUM_THREADS=1
ulimit -s unlimited
ulimit -c 0
/home/b/b382237/.conda/envs/pcpaper_env/bin/python -m distributed.cli.dask_worker tcp://192.168.0.49:56997 --name dummy-name --nthreads 1 --memory-limit 476.84MiB --nworkers 64 --nanny --death-timeout 60

0
Remote dashboard address: http://192.168.0.49:7777
Setup ssh port forwarding: ssh -L 7777:192.168.0.49:7777 username@levante.dkrz.de
Local dashboard address: http://localhost:7777


---
## View D – Bin-resolved spectral budget waterfall

For selected height levels and a short time window, show each process
rate as a function of droplet/crystal diameter.  This is the only view
that reveals *where in the size spectrum* each process acts.

In [3]:
N_WORKERS = 4

# View D plot styling — adjust here to change alphas, linewidths, bar widths, etc.
cfg.update({
    "bar_edge_color": "black",
    "bar_edge_linewidth": 0.35,
    "bar_width_frac_merged": 0.95,
    "pos_ice_alpha": 0.6,
    "pos_liq_alpha": 0.35,
    "neg_ice_alpha": 0.6,
    "neg_liq_alpha": 0.35,
    "ice_hatch": "....",     
    "liq_hatch": None,     
    # "bar_width_factor": 0.12,  # applies when stacked_bars=False
    # "bar_width_ptp_frac": 0.35,  # applies when stacked_bars=False
    "rate_linewidth": 0.5,
    "rate_linewidth_neg": 0.5,
    "rate_fill_linewidth": 0.9,
    "rate_edge_linewidth": 0.2,
    "grid_linewidth": 0.15,
    "grid_alpha": 0.5,
    "zero_linewidth": 0.4,
    "panel_bbox_alpha": 0.35,
})

# Use seed_start, station_labels, height_sel_m from shared (%run). Override time_window for this view:
# time_spacing = np.arange(0.0, 35.0, 30/60)
seed_start = np.datetime64("2023-01-25T12:29:00")
time_spacing = np.arange(0.0, 35.0, 10/60)
time_window = [seed_start + np.timedelta64(int(t * 60), "s") for t in time_spacing]



idx_dbg = 1
rates_plot_config = {               
                    "linthresh_W": 1e-10, 
                    "linthresh_F": 1e-10, 
                    "linscale": 0.1,
                    "xlim_W": (0.001, 6e2), 
                    "xlim_F": (0.001, 6e2), 
                    "ylim_W": (-1e1, 1e1), 
                    "ylim_F": (-1e1, 1e1),
                    "stacked_bars": True, 
                    "merge_liquid_ice": True, 
                    "si": plot_stn_ids, } 


In [ ]:



# Generic helpers for View D spectral processing and stacking
def _pad_1d(a, length, fill_value=0.0):
    """Pad or truncate 1D array to length; fill with fill_value."""
    a = np.asarray(a)
    if a.size >= length:
        return a[:length]
    pad_width = length - a.size
    return np.pad(a, (0, pad_width), constant_values=fill_value)

def _nonneg_from(arr):
    """Return array with NaN/inf replaced and values clipped to >= 0."""
    a = np.asarray(arr)
    a = np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0)
    return np.maximum(0.0, a)

def _nonneg_from_negative(arr):
    """Return -arr clipped to >= 0 (for stacking negative contributions)."""
    a = np.asarray(arr)
    a = np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0)
    return np.maximum(0.0, -a)

def _build_liq_ice_stacks(sp_w, sp_f, n, proc_color):
    """Return procs_pos dict: process -> (color, liq_arr, ice_arr)."""

    sp_w, sp_f = list(sp_w), list(sp_f)
    procs_pos = {}  # p -> (c, liq_pos, ice_pos)
    for p, c, arr in sp_w:
        liq = _nonneg_from(_pad_1d(arr, n))
        procs_pos[p] = [c, liq, np.zeros(n)]
    for p, c, arr in sp_f:
        ice = _nonneg_from(_pad_1d(arr, n))
        if p not in procs_pos:
            procs_pos[p] = [proc_color(p), np.zeros(n), ice]
        else:
            procs_pos[p] = [procs_pos[p][0], procs_pos[p][1], ice]
    return procs_pos

def _build_liq_ice_stacks_neg(sn_w, sn_f, n, proc_color):
    """Return procs_neg dict: process -> (color, liq_arr, ice_arr)."""
    sn_w, sn_f = list(sn_w), list(sn_f)
    procs_neg = {}
    for p, c, arr in sn_w:
        liq = _nonneg_from_negative(_pad_1d(arr, n))
        procs_neg[p] = [c, liq, np.zeros(n)]
    for p, c, arr in sn_f:
        ice = _nonneg_from_negative(_pad_1d(arr, n))
        if p not in procs_neg:
            procs_neg[p] = [proc_color(p), np.zeros(n), ice]
        else:
            procs_neg[p] = [procs_neg[p][0], procs_neg[p][1], ice]
    return procs_neg

def _stack_contrib(items, kind):
    """Return scalar contribution per item for sorting (kind in "pos"|"neg"|"net")."""
    if kind == 'pos':
        return [float(np.nansum(np.maximum(0.0, np.asarray(a)))) for (_, _, a) in items]
    if kind == 'neg':
        return [float(np.nansum(np.maximum(0.0, -np.asarray(a)))) for (_, _, a) in items]
    return [float(np.nansum(np.abs(np.asarray(a)))) for (_, _, a) in items]

def plot_spectral_waterfall(spec_rates_W, spec_rates_F, 
                            unit_label, kind_label,
                            height_sel_m=height_sel_m,
                            time_window=time_window,
                            si=stn_idx, 
                            linthresh_W=1e-12, 
                            linthresh_F=1e-12,
                            linscale=2, 
                            xlim_W=(1e-1, 1e2),
                            xlim_F=(1e-1, 4e2),
                            ylim_W=(-1e4, 1e4),
                            ylim_F=(-1e4, 1e4),
                            spec_rates_W_pos=None, 
                            spec_rates_W_neg=None,
                            spec_rates_F_pos=None, 
                            spec_rates_F_neg=None,
                            range_key_w=None, 
                            range_key_f=None, 
                            stacked_bars=False,
                            merge_liquid_ice=False):
    
    """View D — spectral budget per process; use pos/neg contribution vars when provided.

    When spec_rates_*_pos and spec_rates_*_neg are provided, positive (source) and
    negative (sink) contributions are plotted separately above/below zero to avoid
    net sign flips. Otherwise net from spec_rates_W / spec_rates_F is used.
    stacked_bars=True draws stacked bars per diameter bin instead of step lines. merge_liquid_ice=True uses one column with liquid and ice bars side-by-side per diameter bin. si can be one station index or a list of indices (one column per station).
    """
    from matplotlib.ticker import FuncFormatter

    



    use_pos_neg = (spec_rates_W_pos is not None and spec_rates_W_neg is not None and
                  spec_rates_F_pos is not None and spec_rates_F_neg is not None)
    if use_pos_neg:
        spec_both_pos = {p: spec_rates_W_pos[p] for p in spec_rates_W_pos}
        for p, da in spec_rates_F_pos.items():
            spec_both_pos[p] = spec_both_pos[p] + da if p in spec_both_pos else da
        spec_both_neg = {p: spec_rates_W_neg[p] for p in spec_rates_W_neg}
        for p, da in spec_rates_F_neg.items():
            spec_both_neg[p] = spec_both_neg[p] + da if p in spec_both_neg else da
        _dicts_pos = {"W": spec_rates_W_pos, "F": spec_rates_F_pos, None: spec_both_pos}
        _dicts_neg = {"W": spec_rates_W_neg, "F": spec_rates_F_neg, None: spec_both_neg}

    # Merge W+F keys for the precipitation column (both spectra)
    spec_rates_both = {
        p: spec_rates_W[p] for p in spec_rates_W
    }
    for p, da in spec_rates_F.items():
        spec_rates_both[p] = spec_rates_both[p] + da if p in spec_rates_both else da

    _dicts = {"W": spec_rates_W, "F": spec_rates_F, None: spec_rates_both}

    range_key_w = range_key_w or active_range_key
    range_key_f = range_key_f or active_range_key
    if merge_liquid_ice:
        spec_columns = [
            (size_ranges[range_key_w]["slice"], size_ranges[range_key_f]["slice"],
             f"Liquid + Ice ({range_key_w})", ("W", "F"),
             (min(xlim_W[0], xlim_F[0]), max(xlim_W[1], xlim_F[1])),
             (min(ylim_W[0], ylim_F[0]), max(ylim_W[1], ylim_F[1])),
             max(linthresh_W, linthresh_F)),
        ]
    else:
        spec_columns = [
            (size_ranges[range_key_w]["slice"], f"Liquid ({range_key_w})", "W", xlim_W, ylim_W, linthresh_W),
            (size_ranges[range_key_f]["slice"], f"Ice ({range_key_f})",    "F", xlim_F, ylim_F, linthresh_F),
        ]

    n_hl = len(height_sel_m)-1
    n_cols = len(spec_columns)
    station_list = list(si) if (hasattr(si, "__iter__") and not isinstance(si, str)) else [si]
    if len(station_list) > 1:
        n_cols = len(station_list)

    fig, axes = plt.subplots(n_hl, n_cols,
                             figsize=(SINGLE_COL_IN * max(1, n_cols), min(n_hl * 80 * MM, MAX_H_IN)),
                             constrained_layout=True)
    fig.set_constrained_layout_pads(h_pad=0)  # no vertical gap between subplot rows
    if n_hl == 1:
        axes = axes[np.newaxis, :]
    if n_cols == 1:
        axes = axes[:, np.newaxis]

    for row in range(n_hl):
        h_m0, h_m1 = height_sel_m[row], height_sel_m[row+1]
        for col in range(n_cols):
            si_cur = station_list[col] if len(station_list) > 1 else station_list[0]
            sl_cur = station_labels[si_cur] if len(station_list) > 1 else station_labels[si_cur]
            col_spec = spec_columns[0] if len(station_list) > 1 else spec_columns[col]
            if len(col_spec) == 7:
                bin_slice_w, bin_slice_f, spec_label, (k_w, k_f), xlim, ylim, linthresh = col_spec
                merged = True
                col_specs = [(bin_slice_w, k_w), (bin_slice_f, k_f)]
            else:
                (bin_slice, spec_label, spectrum_key, xlim, ylim, linthresh) = col_spec
                merged = False
                col_specs = [(bin_slice, spectrum_key)]
            ax = axes[row, col]
            any_data = False
            panel_labeled = []
            panel_signal = []
            all_stacks = []
            for (bin_slice, spectrum_key) in col_specs:
                spec_rates = _dicts[spectrum_key]
                procs = list(spec_rates.keys())
                colors = [proc_color(p) for p in procs]
                stack_pos, stack_neg, stack_net = [], [], []
                d = diameter_um[bin_slice]
                for p, c in zip(procs, colors):
                    da = spec_rates[p].isel(station=si_cur).sel(height_level=slice(h_m0, h_m1)).mean(dim="height_level")
                    actual_h = (h_m1- h_m0) / 2
                    da_t = da.sel(time=time_window)
                    if da_t.sizes["time"] == 0:
                        continue
                    if use_pos_neg and p in _dicts_pos[spectrum_key] and p in _dicts_neg[spectrum_key]:
                        da_pos = _dicts_pos[spectrum_key][p].isel(station=si_cur).sel(height_level=slice(h_m0, h_m1)).mean(dim="height_level").sel(time=time_window)
                        da_neg = _dicts_neg[spectrum_key][p].isel(station=si_cur).sel(height_level=slice(h_m0, h_m1)).mean(dim="height_level").sel(time=time_window)
                        pos_vals = da_pos.mean(dim="time").isel(bins=bin_slice).values
                        neg_vals = da_neg.mean(dim="time").isel(bins=bin_slice).values
                    if np.any(np.isfinite(pos_vals) & (pos_vals > 0)):
                        panel_signal.append(p)
                        if stacked_bars:
                            stack_pos.append((p, c, np.asarray(pos_vals)))
                        else:
                            ax.fill_between(d, 0, pos_vals, step="mid", facecolor=(*mcolors.to_rgb(c), 0.25),
                                            edgecolor=(*mcolors.to_rgb(c), 0.9), linewidth=cfg["rate_fill_linewidth"], label=p)
                            panel_labeled.append(p)
                            ax.plot(d, pos_vals, color=c, linewidth=cfg["rate_linewidth"], drawstyle="steps-mid")
                            ax.plot(d, pos_vals, color='black', linewidth=cfg["rate_edge_linewidth"], drawstyle="steps-mid")
                        any_data = True
                    if np.any(np.isfinite(neg_vals) & (neg_vals < 0)):
                        panel_signal.append(p)
                        if stacked_bars:
                            stack_neg.append((p, c, np.asarray(neg_vals)))
                        else:
                            ax.fill_between(d, neg_vals, 0, step="mid", facecolor=(*mcolors.to_rgb(c), 0.25),
                                            edgecolor=(*mcolors.to_rgb(c), 0.9), linewidth=cfg["rate_fill_linewidth"], linestyle="--")
                            ax.plot(d, neg_vals, color=c, linewidth=cfg["rate_linewidth_neg"], drawstyle="steps-mid", linestyle="--")
                            ax.plot(d, neg_vals, color='black', linewidth=cfg["rate_edge_linewidth"], drawstyle="steps-mid")
                        any_data = True
                    else:
                        vals = da_t.mean(dim="time").isel(bins=bin_slice).values
                        if np.any(np.isfinite(vals) & (vals != 0)):
                            panel_signal.append(p)
                            if stacked_bars:
                                stack_net.append((p, c, np.asarray(vals)))
                            else:
                                ax.fill_between(d, 0, vals, step="mid", facecolor=(*mcolors.to_rgb(c), 0.25),
                                                edgecolor=(*mcolors.to_rgb(c), 0.9), linewidth=cfg["rate_fill_linewidth"], label=p)
                                ax.plot(d, vals, color=c, linewidth=cfg["rate_linewidth"], drawstyle="steps-mid")
                                ax.plot(d, vals, color='black', linewidth=cfg["rate_edge_linewidth"], drawstyle="steps-mid")
                            any_data = True
                if merged:
                    all_stacks.append((np.asarray(d), list(stack_pos), list(stack_neg), list(stack_net)))
                    any_data = any_data or bool(stack_pos or stack_neg or stack_net)
            if stacked_bars and (merged and len(all_stacks) == 2 or (stack_pos or stack_neg or stack_net)):
                stack_pos = sorted(stack_pos, key=lambda x: _stack_contrib([x], 'pos')[0])
                stack_neg = sorted(stack_neg, key=lambda x: _stack_contrib([x], 'neg')[0])
                stack_net = sorted(stack_net, key=lambda x: _stack_contrib([x], 'net')[0])
                if merged and len(all_stacks) == 2:
                    # Stack liquid and ice on top of each other per bin (one x per bin)
                    d_w, d_f = all_stacks[0][0], all_stacks[1][0]
                    n = min(len(d_w), len(d_f))
                    d_w, d_f = d_w[:n], d_f[:n]
                    x_single = (d_w + d_f) / 2.0  # one position per bin
                    diff_w = np.diff(d_w) if n > 1 else np.array([d_w[0] * 0.1])
                    bin_width = np.concatenate([diff_w, [diff_w[-1]]]) if n > 1 else np.array([d_w[0] * 0.1])
                    widths = cfg["bar_width_frac_merged"] * bin_width
                    sp_w, sn_w = all_stacks[0][1], all_stacks[0][2]
                    sp_f, sn_f = all_stacks[1][1], all_stacks[1][2]
                    procs_pos = _build_liq_ice_stacks(sp_w, sp_f, n, proc_color)
                    procs_neg = _build_liq_ice_stacks_neg(sn_w, sn_f, n, proc_color)
                    order_pos = sorted(procs_pos.keys(), key=lambda p: np.sum(procs_pos[p][1] + procs_pos[p][2]))
                    order_neg = sorted(procs_neg.keys(), key=lambda p: np.sum(procs_neg[p][1] + procs_neg[p][2]))
                    bottom = np.zeros(n)
                    for p in order_pos:
                        c, liq, ice = procs_pos[p]
                        ax.bar(x_single, ice, width=widths, bottom=bottom, color=c, edgecolor=cfg["bar_edge_color"], linewidth=cfg["bar_edge_linewidth"], alpha=cfg["pos_ice_alpha"], hatch=cfg["ice_hatch"]); bottom += ice
                        ax.bar(x_single, liq, width=widths, bottom=bottom, color=c, label=p, edgecolor=cfg["bar_edge_color"], linewidth=cfg["bar_edge_linewidth"], alpha=cfg["pos_liq_alpha"], hatch=cfg["liq_hatch"]); bottom += liq
                    bottom_neg = np.zeros(n)
                    for p in order_neg:
                        c, liq, ice = procs_neg[p]
                        ax.bar(x_single, ice, width=widths, bottom=-bottom_neg - ice, color=c, edgecolor=cfg["bar_edge_color"], linewidth=cfg["bar_edge_linewidth"], alpha=cfg["neg_ice_alpha"], hatch=cfg["ice_hatch"]); bottom_neg += ice
                        ax.bar(x_single, liq, width=widths, bottom=-bottom_neg - liq, color=c, label=p, edgecolor=cfg["bar_edge_color"], linewidth=cfg["bar_edge_linewidth"], alpha=cfg["neg_liq_alpha"], hatch=cfg["liq_hatch"]); bottom_neg += liq
                else:
                    widths = np.clip(cfg["bar_width_factor"] * d, 1e-10, np.ptp(d) * cfg["bar_width_ptp_frac"])
                    if stack_pos:
                        bottom = np.zeros_like(d, dtype=float)
                        for p, c, arr in stack_pos:
                            arr = np.where(np.isfinite(arr), np.maximum(0, arr), 0)
                            ax.bar(d, arr, width=widths, bottom=bottom, color=c, label=p, edgecolor=cfg["bar_edge_color"], linewidth=cfg["bar_edge_linewidth"]); bottom = bottom + arr
                    if stack_neg:
                        bottom_neg = np.zeros_like(d, dtype=float)
                        for p, c, arr in stack_neg:
                            height = np.where(np.isfinite(arr), np.maximum(0, -np.asarray(arr)), 0)
                            ax.bar(d, height, width=widths, bottom=-bottom_neg - height, color=c, label=p, edgecolor=cfg["bar_edge_color"], linewidth=cfg["bar_edge_linewidth"]); bottom_neg += height
                    # if stack_net and not (stack_pos or stack_neg):
                    #     bottom_pos = np.zeros_like(d, dtype=float)
                    #     bottom_neg = np.zeros_like(d, dtype=float)
                    #     for p, c, arr in stack_net:
                    #         arr = np.asarray(arr)
                    #         pos = np.where(np.isfinite(arr), np.maximum(0, arr), 0)
                    #         neg = np.where(np.isfinite(arr), np.maximum(0, -arr), 0)
                    #         ax.bar(d, pos, width=widths, bottom=bottom_pos, color=c, label=p, edgecolor=cfg["bar_edge_color"], linewidth=cfg["bar_edge_linewidth"]); bottom_pos += pos
                    #         ax.bar(d, neg, width=widths, bottom=-bottom_neg - neg, color=c, edgecolor=cfg["bar_edge_color"], linewidth=cfg["bar_edge_linewidth"]); bottom_neg += neg
                any_data = True
            ax.set_xlim(*xlim)
            ax.set_ylim(*ylim)
            ax.set_xscale("log")
            ax.set_yscale("symlog", linthresh=linthresh, linscale=linscale)
            ax.axhline(0, color="grey", linewidth=cfg["zero_linewidth"], linestyle="--")
            ax.grid(True, which="major", linestyle="--", linewidth=cfg["grid_linewidth"],
                    color="k", alpha=cfg["grid_alpha"])
            ax.set_axisbelow(True)
            ax.tick_params(which="both", direction="out")
            ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:g}"))
            from matplotlib.ticker import FixedLocator
            yticks = ax.get_yticks()
            yticklabels = [f"{ytick:g}" if i % 2 == 0 else "" for i, ytick in enumerate(yticks)]
            ax.yaxis.set_major_locator(FixedLocator(yticks))
            ax.set_yticklabels(yticklabels)
            ax.yaxis.set_ticks_position("both")
            ax.yaxis.set_tick_params(right=True, which='both')
            if row == n_hl - 1:
                ax.set_xlabel("Diameter [µm]")
            elif row == 0:
                ax.set_title(stn_label(si_cur, station_labels) if len(station_list) > 1 else spec_label)
            else:
                ax.set_xticklabels([])
            if col == 0:
                ax.set_ylabel(f"Rate [{unit_label}]")
            else:
                ax.set_yticklabels([])
            panel_txt = f"{h_m1:.0f} – {h_m0:.0f} m"
            ax.text(0.95, 0.95, panel_txt, transform=ax.transAxes, ha="right", va="top", fontweight="semibold",
                    bbox=dict(facecolor="white", edgecolor="white", alpha=cfg["panel_bbox_alpha"], boxstyle="round,pad=0.05"))
            if not any_data:
                ax.text(0.5, 0.5, "no signal", transform=ax.transAxes, ha="center", va="center", color="grey")
    legend_by_label = {}
    for ax in axes.flatten():
        ax.spines["top"].set_visible(False)
        # ax.spines["right"].set_visible(False)
        ax.spines["bottom"].set_visible(False)
        h_ax, l_ax = ax.get_legend_handles_labels()
        for h, lbl in zip(h_ax, l_ax):
            if lbl and lbl not in legend_by_label:
                legend_by_label[lbl] = h
    if legend_by_label:
        fig.legend(list(legend_by_label.values()), list(legend_by_label.keys()),
                   loc="lower center", bbox_to_anchor=(0.5, -0.02), ncol=min(6, max(1, len(legend_by_label))), frameon=False)
    tw_str = str(time_window.start)[11:19] + " – " + str(time_window.stop)[11:19] if hasattr(time_window, "start") else ""
    fig.suptitle(f"View D — {kind_label} spectral budget [{unit_label}]  --  {tw_str}", fontweight="semibold")
    return fig, axes


def _render_one_frame(args):
    """Render one (eid, range_key, itime_win) figure; save to output/05/; no plt.show(). Used by parallel or serial loop."""
    eid, range_key, itime_win = args
    R = rates_by_exp[eid]
    fig, axes = plot_spectral_waterfall(
        R["spec_rates_N_W"], 
        R["spec_rates_N_F"], 
        R["unit_N"], 
        "Number",
        spec_rates_W_pos=R["spec_rates_N_W_pos"], 
        spec_rates_W_neg=R["spec_rates_N_W_neg"],
        spec_rates_F_pos=R["spec_rates_N_F_pos"], 
        spec_rates_F_neg=R["spec_rates_N_F_neg"],
        range_key_w=range_key, 
        range_key_f=range_key,
        time_window=slice(time_window[itime_win], 
        time_window[itime_win + 1]),
        **rates_plot_config
    )
    save_fig(fig, f"spectral_waterfall_number_exp{eid}_stn_all_{range_key}_itime{itime_win}", "png", "./output/05/")
    plt.close(fig)
    return (eid, range_key, itime_win)


# MAIN: generate all figures (parallel with serial fallback)
if "rates_by_exp" in dir() and rates_by_exp:
    tasks = [(eid, range_key, itime_win)
             for eid in plot_exp_ids[idx_dbg:idx_dbg+1]
             for range_key in plot_range_keys[-idx_dbg:]
             for itime_win in range(len(time_window) - 1)]
    use_parallel = len(tasks) > 1
    ran = False
    print(f"Generating {len(tasks)} frames")
    print(tasks)
    print("\n\n")
    if use_parallel:
        import os
        import warnings
        matplotlib.use("Agg")
        # Prefer ProcessPoolExecutor (true multi-core); use loky so notebook _render_one_frame is picklable
        ProcessPoolExecutor = None
        try:
            from loky import ProcessPoolExecutor
        except ImportError:
            pass
        try:
            available_workers = len(os.sched_getaffinity(0))
        except AttributeError:
            available_workers = os.cpu_count() or 1
        
        print(f"Available workers: {available_workers}")
        max_workers = min(len(tasks), available_workers)
        
        if ProcessPoolExecutor is not None:
            try:
                print(f"Using ProcessPoolExecutor with {max_workers} workers")
                with ProcessPoolExecutor(max_workers=max_workers) as executor:
                    list(executor.map(_render_one_frame, tasks))
                ran = True
            except Exception as e:
                warnings.warn(f"ProcessPoolExecutor failed ({e}), trying threads then serial.")
        if use_parallel:
            try:
                from concurrent.futures import ThreadPoolExecutor
                print(f"Using ThreadPoolExecutor with {max_workers} workers")
                with ThreadPoolExecutor(max_workers=max_workers) as executor:
                    list(executor.map(_render_one_frame, tasks))
                ran = True
            except Exception as e:
                warnings.warn(f"ThreadPoolExecutor failed ({e}), falling back to serial.")
    if not ran:
        for args in tasks:
            _render_one_frame(args)



Generating 209 frames
[(2, 'ALLBB', 0), (2, 'ALLBB', 1), (2, 'ALLBB', 2), (2, 'ALLBB', 3), (2, 'ALLBB', 4), (2, 'ALLBB', 5), (2, 'ALLBB', 6), (2, 'ALLBB', 7), (2, 'ALLBB', 8), (2, 'ALLBB', 9), (2, 'ALLBB', 10), (2, 'ALLBB', 11), (2, 'ALLBB', 12), (2, 'ALLBB', 13), (2, 'ALLBB', 14), (2, 'ALLBB', 15), (2, 'ALLBB', 16), (2, 'ALLBB', 17), (2, 'ALLBB', 18), (2, 'ALLBB', 19), (2, 'ALLBB', 20), (2, 'ALLBB', 21), (2, 'ALLBB', 22), (2, 'ALLBB', 23), (2, 'ALLBB', 24), (2, 'ALLBB', 25), (2, 'ALLBB', 26), (2, 'ALLBB', 27), (2, 'ALLBB', 28), (2, 'ALLBB', 29), (2, 'ALLBB', 30), (2, 'ALLBB', 31), (2, 'ALLBB', 32), (2, 'ALLBB', 33), (2, 'ALLBB', 34), (2, 'ALLBB', 35), (2, 'ALLBB', 36), (2, 'ALLBB', 37), (2, 'ALLBB', 38), (2, 'ALLBB', 39), (2, 'ALLBB', 40), (2, 'ALLBB', 41), (2, 'ALLBB', 42), (2, 'ALLBB', 43), (2, 'ALLBB', 44), (2, 'ALLBB', 45), (2, 'ALLBB', 46), (2, 'ALLBB', 47), (2, 'ALLBB', 48), (2, 'ALLBB', 49), (2, 'ALLBB', 50), (2, 'ALLBB', 51), (2, 'ALLBB', 52), (2, 'ALLBB', 53), (2, 'ALLBB', 54

***Figure — Bin-resolved spectral budget.** Time-averaged microphysical process rates are shown as a function of equivalent particle diameter for short moving windows around seeding onset. Each row corresponds to a model height layer (1200–1400 m, 1000–1200 m, 800–1000 m), and each column corresponds to one station (S1 seeding location, S2 observation site, S3 precipitation location). Within each panel, liquid and ice contributions are merged and plotted as stacked bars by diameter bin, with positive source terms above zero and negative sink terms below zero on a symlog y-axis. This view preserves the full spectral axis and reveals where in the particle-size distribution each microphysical process acts. The current routine produces a sequence of number-rate frames, which are then combined into an animated GIF to show the temporal evolution around seeding.*

In [ ]:
import glob
from PIL import Image

# MAIN
if "rates_by_exp" in dir() and rates_by_exp:
    for eid in plot_exp_ids[idx_dbg:idx_dbg+1]:
        R = rates_by_exp[eid]
        exp_label = R["exp_label"]

        for range_key in plot_range_keys[-idx_dbg:]:
            # After generating all frames for this experiment and range, create a GIF
            frame_pattern = f"./output/05/spectral_waterfall_number_exp{eid}_stn_all_{range_key}_itime*.png"
            frames = glob.glob(frame_pattern)
            if frames:
                frames.sort(key=lambda x: int(x.split('_itime')[-1].split('.png')[0]))
                images = [Image.open(frame) for frame in frames]
                gif_path = f"./output/05/spectral_waterfall_number_exp{eid}_stn_all_{range_key}_evolution.gif"
                images[0].save(
                    gif_path, 
                    save_all=True, 
                    append_images=images[1:], 
                    duration=800, 
                    loop=0
                )
                print(f"Created animated GIF: {gif_path}")


Created animated GIF: ./output/05/spectral_waterfall_number_exp2_stn_all_ALLBB_evolution.gif


In [ ]:

c1.close()
c2.close()

In [3]:
!pip install "dask[distributed]" --upgrade

In [ ]:
rates_by_exp[plot_exp_ids[0]]['spec_rates_N_W']

{'BREAKUP': <xarray.DataArray (time: 4033, station: 3, height_level: 20, bins: 66)> Size: 128MB
 dask.array<add, shape=(4033, 3, 20, 66), dtype=float64, chunksize=(499, 3, 20, 66), chunktype=numpy.ndarray>
 Coordinates:
     station_lat     (station) float64 24B dask.array<chunksize=(3,), meta=np.ndarray>
     radius_centers  (bins) float64 528B dask.array<chunksize=(66,), meta=np.ndarray>
     HMLd            (height_level) float32 80B dask.array<chunksize=(20,), meta=np.ndarray>
     station_lon     (station) float64 24B dask.array<chunksize=(3,), meta=np.ndarray>
   * station         (station) int32 12B 1 2 3
     expname         |S14 14B b'20260304110638'
     mass_centers    (bins) float64 528B dask.array<chunksize=(66,), meta=np.ndarray>
   * height_level    (height_level) float32 80B 1.48e+03 1.417e+03 ... 813.3
   * time            (time) datetime64[ns] 32kB 2023-01-25T12:00:00 ... 2023-0...
 Dimensions without coordinates: bins
 Attributes:
     standard_name:  accumulated ten